
# Water Potability — Kernel cuántico y estudio de mapas de características (Partes 3 y 4)

Este notebook continúa desde los **64 ejemplos reales y los cuatro folds congelados** generados en la etapa clásica. No vuelve a seleccionar observaciones y no utiliza el holdout completo.

## Objetivos

1. Construir un mapa cuántico $|\phi(x)\rangle$ con Qiskit.
2. Calcular explícitamente
   $$K_{ij}=|\langle\phi(x_i)|\phi(x_j)\rangle|^2.$$
3. Entrenar `sklearn.svm.SVC(kernel="precomputed")`.
4. Entregar el diagrama del circuito y el mapa de calor del kernel.
5. Comparar `ZZFeatureMap` y `PauliFeatureMap` mediante:
   - accuracy, balanced accuracy y F1 con validación cruzada;
   - alineación kernel–objetivo;
   - similitudes intra e interclase;
   - espectro y rango efectivo;
   - profundidad y compuertas de dos qubits;
   - sensibilidad a shots, ruido y topología;
   - costo computacional;
   - ablaciones de repeticiones, entrelazamiento, términos de Pauli y escalado.

## Regla metodológica

En cada fold, la imputación y el escalado se ajustan **solo con las 48 muestras de entrenamiento**. Las 16 observaciones de validación permanecen fuera de todo ajuste. El parámetro `C` se selecciona mediante una validación interna sobre el kernel de entrenamiento.

## Número de qubits

El experimento principal utiliza **9 qubits**, uno por cada variable fisicoquímica. No se aplica PCA en el análisis principal, para que el mapa cuántico reciba exactamente las mismas nueve características originales.


In [ ]:

# 1. Drive, versiones e importaciones
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Versiones fijadas para reproducibilidad.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'qiskit==2.2.3',
    'qiskit-aer==0.17.2',
    'pylatexenc'
], check=True)

import json
import hashlib
import time
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.quantum_info import Statevector
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

warnings.filterwarnings('ignore', category=FutureWarning)
np.set_printoptions(precision=4, suppress=True)

SEED = 42
NUM_QUBITS = 9
C_GRID = (0.1, 1.0, 10.0)
QUICK_MODE = False            # True: ejecuta solo tres mapas para una prueba rápida.
RUN_SHOT_STUDY = True
RUN_NOISE_STUDY = True
SHOT_LEVELS = (128, 512, 2048)
SHOT_REPEATS = 5
NOISE_SHOTS = 1024
NOISE_PAIRS_PER_MAP = 12

if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/Colab Notebooks')
else:
    BASE_DIR = Path('/mnt/data')

DATA_DIR = BASE_DIR / 'artifacts_qsvm_input_v2'
ARTIFACT_DIR = BASE_DIR / 'artifacts_quantum_kernel_v1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Qubits:', NUM_QUBITS)
print('Artifacts:', ARTIFACT_DIR)


In [ ]:

# 2. Carga y validación de los 64 datos y folds congelados
raw_candidates = [
    DATA_DIR / 'qsvm_64_raw.csv',
    BASE_DIR / 'qsvm_64_raw.csv',
    BASE_DIR / 'svm64_training_raw.csv',
]
fold_candidates = [
    DATA_DIR / 'qsvm_64_folds.csv',
    BASE_DIR / 'qsvm_64_folds.csv',
    BASE_DIR / 'svm64_fixed_cv_folds.csv',
]

raw_path = next((p for p in raw_candidates if p.exists()), None)
fold_path = next((p for p in fold_candidates if p.exists()), None)
if raw_path is None or fold_path is None:
    raise FileNotFoundError('No se encontraron qsvm_64_raw.csv y qsvm_64_folds.csv.')

raw64 = pd.read_csv(raw_path)
folds64 = pd.read_csv(fold_path)

FEATURES = [
    'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
    'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'
]
TARGET = 'Potability'

assert raw64.shape[0] == 64
assert raw64[TARGET].value_counts().to_dict() == {0: 32, 1: 32}
assert set(folds64['validation_fold'].unique()) == {0, 1, 2, 3}
assert raw64['source_index'].tolist() == folds64['source_index'].tolist()
assert raw64[TARGET].tolist() == folds64[TARGET].tolist()
assert len(FEATURES) == NUM_QUBITS

X_raw = raw64[FEATURES].to_numpy(dtype=float)
y = raw64[TARGET].to_numpy(dtype=int)
validation_fold = folds64['validation_fold'].to_numpy(dtype=int)

fold_summary = (
    folds64.groupby(['validation_fold', TARGET])
    .size().unstack(fill_value=0)
    .rename(columns={0: 'class_0', 1: 'class_1'})
)
fold_summary['validation_size'] = fold_summary.sum(axis=1)
fold_summary['training_size'] = 64 - fold_summary['validation_size']

display(fold_summary)
print('Valores faltantes por variable:')
display(raw64[FEATURES].isna().sum().to_frame('missing'))



## Preprocesamiento angular sin fuga

Los circuitos de características reciben ángulos. En cada fold se ajusta una imputación mediana y una transformación angular usando únicamente el training:

- `minmax`: proyecta cada variable a $[0,\pi]$ y recorta valores externos del validation.
- `stdclip`: estandariza, recorta a $[-3,3]$ y transforma a $[0,\pi]$.
- `atan`: estandariza y usa $\arctan(z)+\pi/2$, que también queda en $(0,\pi)$.


In [ ]:

# 3. Preprocesamiento, mapas cuánticos y kernel exacto
@dataclass
class AnglePreprocessor:
    mode: str = 'minmax'

    def fit(self, X):
        self.imputer_ = SimpleImputer(strategy='median')
        Xi = self.imputer_.fit_transform(X)
        if self.mode == 'minmax':
            self.scaler_ = MinMaxScaler(feature_range=(0.0, np.pi)).fit(Xi)
        elif self.mode in {'stdclip', 'atan'}:
            self.scaler_ = StandardScaler().fit(Xi)
        else:
            raise ValueError(f'Modo no soportado: {self.mode}')
        return self

    def transform(self, X):
        Xi = self.imputer_.transform(X)
        Z = self.scaler_.transform(Xi)
        if self.mode == 'minmax':
            return np.clip(Z, 0.0, np.pi)
        if self.mode == 'stdclip':
            Z = np.clip(Z, -3.0, 3.0)
            return (Z + 3.0) * np.pi / 6.0
        return np.arctan(Z) + np.pi / 2.0

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def build_feature_map(config):
    common = dict(
        feature_dimension=NUM_QUBITS,
        reps=int(config['reps']),
        entanglement=config['entanglement'],
        insert_barriers=False,
    )
    if config['family'] == 'ZZ':
        return ZZFeatureMap(**common)
    return PauliFeatureMap(paulis=list(config['paulis']), **common)


def state_matrix(feature_map, X_angles):
    """Devuelve una matriz N x 2^n con los statevectors |phi(x)> como filas."""
    params = list(feature_map.parameters)
    if len(params) != X_angles.shape[1]:
        raise ValueError('El número de parámetros no coincide con las características.')

    states = []
    for row in X_angles:
        bound = feature_map.assign_parameters(dict(zip(params, row)), inplace=False)
        states.append(Statevector.from_instruction(bound).data)
    return np.asarray(states, dtype=complex)


def fidelity_kernel_from_states(A, B=None):
    """K_ij = |<phi(x_i)|phi(x_j)>|^2."""
    if B is None:
        B = A
    overlaps = A.conj() @ B.T
    K = np.abs(overlaps) ** 2
    return np.clip(K.real, 0.0, 1.0)


def exact_kernel(feature_map, X_left, X_right=None):
    A = state_matrix(feature_map, X_left)
    B = A if X_right is None else state_matrix(feature_map, X_right)
    K = fidelity_kernel_from_states(A, B)
    if X_right is None:
        K = (K + K.T) / 2.0
        np.fill_diagonal(K, 1.0)
    return K


def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def centered_kernel_alignment(K, labels):
    n = len(labels)
    H = np.eye(n) - np.ones((n, n)) / n
    y_pm = 2 * np.asarray(labels) - 1
    Y = np.outer(y_pm, y_pm)
    Kc = H @ K @ H
    Yc = H @ Y @ H
    den = np.linalg.norm(Kc, 'fro') * np.linalg.norm(Yc, 'fro')
    return float(np.sum(Kc * Yc) / den) if den > 0 else np.nan


def similarity_summary(K, labels):
    labels = np.asarray(labels)
    iu = np.triu_indices_from(K, k=1)
    same = labels[iu[0]] == labels[iu[1]]
    values = K[iu]
    intra = float(values[same].mean())
    inter = float(values[~same].mean())
    return intra, inter, intra - inter


def effective_rank(K, tol=1e-12):
    eig = np.linalg.eigvalsh((K + K.T) / 2.0)
    eig = np.clip(eig, 0.0, None)
    positive = eig[eig > tol]
    if positive.size == 0:
        return 0.0, eig
    p = positive / positive.sum()
    return float(np.exp(-(p * np.log(p)).sum())), eig


def choose_c_inner(K_train, y_train, seed):
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    rows = []
    for C in C_GRID:
        scores = []
        for tr, va in inner.split(np.zeros(len(y_train)), y_train):
            model = SVC(kernel='precomputed', C=C)
            model.fit(K_train[np.ix_(tr, tr)], y_train[tr])
            pred = model.predict(K_train[np.ix_(va, tr)])
            scores.append(f1_score(y_train[va], pred, zero_division=0))
        rows.append((C, float(np.mean(scores))))
    return max(rows, key=lambda item: (item[1], -item[0]))[0], rows



# Parte 3 — Kernel cuántico base

Se utiliza como mapa inicial:

```python
ZZFeatureMap(feature_dimension=9, reps=1, entanglement="linear")
```

Cada observación produce un estado de 9 qubits, es decir, un vector complejo en un espacio de dimensión $2^9=512$. El kernel no necesita almacenar explícitamente una expansión clásica de 512 variables: calcula directamente la fidelidad entre pares de estados.


In [ ]:

# 4. Configuraciones del estudio
MAP_CONFIGS = [
    dict(name='ZZ_r1_linear_minmax', family='ZZ', reps=1, entanglement='linear', paulis=(), scaling='minmax'),
    dict(name='ZZ_r2_linear_minmax', family='ZZ', reps=2, entanglement='linear', paulis=(), scaling='minmax'),
    dict(name='ZZ_r1_full_minmax', family='ZZ', reps=1, entanglement='full', paulis=(), scaling='minmax'),
    dict(name='Pauli_Z_ZZ_r1_linear_minmax', family='Pauli', reps=1, entanglement='linear', paulis=('Z', 'ZZ'), scaling='minmax'),
    dict(name='Pauli_Z_ZZ_r2_linear_minmax', family='Pauli', reps=2, entanglement='linear', paulis=('Z', 'ZZ'), scaling='minmax'),
    dict(name='Pauli_Z_ZZ_r1_full_minmax', family='Pauli', reps=1, entanglement='full', paulis=('Z', 'ZZ'), scaling='minmax'),
    dict(name='Pauli_X_Y_ZZ_r1_linear_minmax', family='Pauli', reps=1, entanglement='linear', paulis=('X', 'Y', 'ZZ'), scaling='minmax'),
    dict(name='ZZ_r1_linear_stdclip', family='ZZ', reps=1, entanglement='linear', paulis=(), scaling='stdclip'),
    dict(name='ZZ_r1_linear_atan', family='ZZ', reps=1, entanglement='linear', paulis=(), scaling='atan'),
]

if QUICK_MODE:
    MAP_CONFIGS = [MAP_CONFIGS[i] for i in (0, 1, 3)]

BASELINE_NAME = 'ZZ_r1_linear_minmax'
config_by_name = {c['name']: c for c in MAP_CONFIGS}

display(pd.DataFrame(MAP_CONFIGS))


In [ ]:

# 5. Diagrama y kernel global descriptivo del mapa base
baseline_cfg = config_by_name[BASELINE_NAME]
baseline_map = build_feature_map(baseline_cfg)

print(baseline_map)
print('Parámetros:', len(baseline_map.parameters))
print('Qubits:', baseline_map.num_qubits)
print('Dimensión del espacio de estados:', 2 ** baseline_map.num_qubits)

# Diagrama del circuito real de 9 qubits.
fig = baseline_map.decompose().draw(output='mpl', fold=120)
fig.savefig(ARTIFACT_DIR / 'baseline_feature_map_circuit.png', dpi=180, bbox_inches='tight')
plt.show()

# Esta matriz global es descriptiva; las métricas predictivas se calculan después
# con preprocesamiento separado dentro de cada fold.
global_pre = AnglePreprocessor(baseline_cfg['scaling'])
X_global = global_pre.fit_transform(X_raw)
start = time.perf_counter()
K_global_baseline = exact_kernel(baseline_map, X_global)
baseline_global_kernel_seconds = time.perf_counter() - start

order = np.argsort(y)
plt.figure(figsize=(7, 6))
plt.imshow(K_global_baseline[np.ix_(order, order)], interpolation='nearest', aspect='auto')
plt.colorbar(label='Fidelidad')
plt.title('Kernel exacto — ZZFeatureMap, muestras ordenadas por clase')
plt.xlabel('Muestra')
plt.ylabel('Muestra')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'baseline_kernel_heatmap.png', dpi=180)
plt.show()

print(f'Tiempo del kernel global base: {baseline_global_kernel_seconds:.3f} s')
print('Diagonal mínima/máxima:', np.diag(K_global_baseline).min(), np.diag(K_global_baseline).max())


In [ ]:

# 6. Validación cruzada exacta con SVC(kernel="precomputed")
def evaluate_config_cv(config):
    fold_rows = []
    cached = {}
    fmap = build_feature_map(config)

    for fold in sorted(np.unique(validation_fold)):
        test_idx = np.where(validation_fold == fold)[0]
        train_idx = np.where(validation_fold != fold)[0]

        pre = AnglePreprocessor(config['scaling'])
        X_train = pre.fit_transform(X_raw[train_idx])
        X_test = pre.transform(X_raw[test_idx])
        y_train, y_test = y[train_idx], y[test_idx]

        t0 = time.perf_counter()
        states_train = state_matrix(fmap, X_train)
        states_test = state_matrix(fmap, X_test)
        state_seconds = time.perf_counter() - t0

        t1 = time.perf_counter()
        K_train = fidelity_kernel_from_states(states_train)
        K_train = (K_train + K_train.T) / 2.0
        np.fill_diagonal(K_train, 1.0)
        K_test = fidelity_kernel_from_states(states_test, states_train)
        kernel_seconds = time.perf_counter() - t1

        best_C, inner_scores = choose_c_inner(K_train, y_train, SEED + fold)
        t2 = time.perf_counter()
        model = SVC(kernel='precomputed', C=best_C)
        model.fit(K_train, y_train)
        pred = model.predict(K_test)
        svc_seconds = time.perf_counter() - t2

        metrics = binary_metrics(y_test, pred)
        intra, inter, gap = similarity_summary(K_train, y_train)
        erank, _ = effective_rank(K_train)
        metrics.update({
            'map': config['name'],
            'fold': int(fold),
            'C': float(best_C),
            'alignment': centered_kernel_alignment(K_train, y_train),
            'intra_similarity': intra,
            'inter_similarity': inter,
            'similarity_gap': gap,
            'effective_rank_train': erank,
            'state_seconds': state_seconds,
            'kernel_seconds': kernel_seconds,
            'svc_seconds': svc_seconds,
            'total_seconds': state_seconds + kernel_seconds + svc_seconds,
        })
        fold_rows.append(metrics)
        cached[fold] = {
            'train_idx': train_idx, 'test_idx': test_idx,
            'y_train': y_train, 'y_test': y_test,
            'K_train': K_train, 'K_test': K_test,
            'C': best_C,
        }

    return pd.DataFrame(fold_rows), cached

all_fold_results = []
kernel_cache = {}

for number, cfg in enumerate(MAP_CONFIGS, start=1):
    print(f'[{number}/{len(MAP_CONFIGS)}] {cfg["name"]}')
    fold_df, cache = evaluate_config_cv(cfg)
    all_fold_results.append(fold_df)
    kernel_cache[cfg['name']] = cache

fold_results = pd.concat(all_fold_results, ignore_index=True)

metric_cols = [
    'accuracy', 'balanced_accuracy', 'f1', 'precision', 'recall',
    'alignment', 'intra_similarity', 'inter_similarity', 'similarity_gap',
    'effective_rank_train', 'total_seconds'
]
summary = fold_results.groupby('map')[metric_cols].agg(['mean', 'std'])
summary.columns = ['_'.join(col) for col in summary.columns]
summary = summary.reset_index().sort_values('f1_mean', ascending=False)

display(summary)
print('Mejor mapa por F1 medio:', summary.iloc[0]['map'])



# Parte 4 — Geometría inducida por el mapa

La exactitud no describe por sí sola la geometría del kernel. Para cada configuración se calcula también:

- **Alineación kernel–objetivo:** similitud normalizada entre el kernel centrado y $yy^T$.
- **Similitud intraclase:** fidelidad promedio entre pares de la misma clase.
- **Similitud interclase:** fidelidad promedio entre clases diferentes.
- **Brecha intra–inter:** valores positivos indican una estructura más favorable.
- **Rango efectivo:** número continuo de direcciones espectrales relevantes. Un kernel casi identidad suele tener rango efectivo alto pero poca estructura compartida; uno casi uniforme concentra casi toda su masa en un solo autovalor.


In [ ]:

# 7. Matrices globales, distribuciones y espectros
# Estas matrices se usan solo para visualización y diagnóstico geométrico.
global_kernels = {}
geometry_rows = []

for cfg in MAP_CONFIGS:
    fmap = build_feature_map(cfg)
    pre = AnglePreprocessor(cfg['scaling'])
    X_angles = pre.fit_transform(X_raw)

    t0 = time.perf_counter()
    K = exact_kernel(fmap, X_angles)
    elapsed = time.perf_counter() - t0
    global_kernels[cfg['name']] = K

    intra, inter, gap = similarity_summary(K, y)
    erank, eig = effective_rank(K)
    geometry_rows.append({
        'map': cfg['name'],
        'global_alignment': centered_kernel_alignment(K, y),
        'global_intra_similarity': intra,
        'global_inter_similarity': inter,
        'global_similarity_gap': gap,
        'global_effective_rank': erank,
        'largest_eigenvalue': float(eig.max()),
        'kernel_seconds_global': elapsed,
    })

    order = np.argsort(y)
    plt.figure(figsize=(6.5, 5.5))
    plt.imshow(K[np.ix_(order, order)], interpolation='nearest', aspect='auto')
    plt.colorbar(label='Fidelidad')
    plt.title(cfg['name'])
    plt.xlabel('Muestras ordenadas por clase')
    plt.ylabel('Muestras ordenadas por clase')
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / f'heatmap_{cfg["name"]}.png', dpi=150)
    plt.show()

    eig_sorted = np.sort(np.clip(eig, 0.0, None))[::-1]
    plt.figure(figsize=(7, 4))
    plt.plot(np.arange(1, len(eig_sorted) + 1), eig_sorted, marker='o', markersize=3)
    plt.yscale('log')
    plt.title(f'Espectro — {cfg["name"]}')
    plt.xlabel('Índice del autovalor')
    plt.ylabel('Autovalor (escala log)')
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / f'spectrum_{cfg["name"]}.png', dpi=150)
    plt.show()

geometry_results = pd.DataFrame(geometry_rows).sort_values('global_alignment', ascending=False)
display(geometry_results)


In [ ]:

# 8. Profundidad, compuertas de dos qubits y sensibilidad a topología
BASIS_GATES = ['rz', 'sx', 'x', 'cx']
topologies = {
    'all_to_all': CouplingMap.from_full(NUM_QUBITS),
    'ring': CouplingMap.from_ring(NUM_QUBITS),
    'line': CouplingMap.from_line(NUM_QUBITS),
}

resource_rows = []
representative_raw = np.nanmedian(X_raw, axis=0, keepdims=True)

for cfg in MAP_CONFIGS:
    pre = AnglePreprocessor(cfg['scaling']).fit(X_raw)
    representative = pre.transform(representative_raw)[0]
    fmap = build_feature_map(cfg)
    params = list(fmap.parameters)
    bound = fmap.assign_parameters(dict(zip(params, representative)), inplace=False)

    for topology_name, coupling in topologies.items():
        t0 = time.perf_counter()
        pm = generate_preset_pass_manager(
            optimization_level=1,
            basis_gates=BASIS_GATES,
            coupling_map=coupling,
            seed_transpiler=SEED,
        )
        physical = pm.run(bound)
        transpile_seconds = time.perf_counter() - t0
        counts = physical.count_ops()
        resource_rows.append({
            'map': cfg['name'],
            'topology': topology_name,
            'depth': int(physical.depth()),
            'cx_count': int(counts.get('cx', 0)),
            'total_gate_count': int(sum(counts.values())),
            'transpile_seconds': transpile_seconds,
        })

resource_results = pd.DataFrame(resource_rows)
display(resource_results.sort_values(['map', 'topology']))



## Sensibilidad a muestreo finito

En un circuito `compute–uncompute`, cada entrada del kernel es la probabilidad de medir $|0\cdots0\rangle$. Para estudiar shots sin repetir miles de simulaciones de circuitos, se realiza el muestreo Bernoulli equivalente:

$$\widehat K_{ij}=\frac{\mathrm{Binomial}(N_{shots}, K_{ij})}{N_{shots}}.$$

La clasificación usa el mismo `C` elegido con el kernel exacto; por tanto, esta sección mide únicamente la degradación introducida por el muestreo.


In [ ]:

# 9. Sensibilidad de cada mapa a shots finitos
def nearest_psd_correlation(K, eps=1e-10):
    K = (K + K.T) / 2.0
    eigvals, eigvecs = np.linalg.eigh(K)
    eigvals = np.clip(eigvals, eps, None)
    Kp = (eigvecs * eigvals) @ eigvecs.T
    d = np.sqrt(np.clip(np.diag(Kp), eps, None))
    Kp = Kp / np.outer(d, d)
    np.fill_diagonal(Kp, 1.0)
    return np.clip(Kp, 0.0, 1.0)


def sample_train_kernel(K, shots, rng):
    n = K.shape[0]
    sampled = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            value = rng.binomial(shots, np.clip(K[i, j], 0, 1)) / shots
            sampled[i, j] = sampled[j, i] = value
    return nearest_psd_correlation(sampled)


def sample_rectangular_kernel(K, shots, rng):
    return rng.binomial(shots, np.clip(K, 0, 1)) / shots

shot_rows = []
if RUN_SHOT_STUDY:
    for map_number, cfg in enumerate(MAP_CONFIGS):
        for shots in SHOT_LEVELS:
            for repeat in range(SHOT_REPEATS):
                rng = np.random.default_rng(SEED + 10000 * map_number + 100 * shots + repeat)
                for fold, item in kernel_cache[cfg['name']].items():
                    Ktr = sample_train_kernel(item['K_train'], shots, rng)
                    Kte = sample_rectangular_kernel(item['K_test'], shots, rng)
                    model = SVC(kernel='precomputed', C=item['C'])
                    model.fit(Ktr, item['y_train'])
                    pred = model.predict(Kte)
                    metrics = binary_metrics(item['y_test'], pred)
                    metrics.update({
                        'map': cfg['name'], 'shots': shots,
                        'repeat': repeat, 'fold': fold,
                        'relative_train_kernel_error': float(
                            np.linalg.norm(Ktr - item['K_train'], 'fro') /
                            np.linalg.norm(item['K_train'], 'fro')
                        ),
                    })
                    shot_rows.append(metrics)

shot_results = pd.DataFrame(shot_rows)
if not shot_results.empty:
    shot_summary = (
        shot_results.groupby(['map', 'shots'])[
            ['accuracy', 'balanced_accuracy', 'f1', 'relative_train_kernel_error']
        ].agg(['mean', 'std']).reset_index()
    )
    shot_summary.columns = [
        '_'.join(c).rstrip('_') if isinstance(c, tuple) else c
        for c in shot_summary.columns
    ]
    display(shot_summary)



## Sensibilidad a ruido

La siguiente ablación usa circuitos explícitos `compute–uncompute` y `AerSimulator`. Para cada mapa se seleccionan pares intra e interclase y se compara la fidelidad exacta con dos niveles sintéticos de ruido despolarizante:

- leve: $p_1=0.001$, $p_2=0.01$;
- moderado: $p_1=0.003$, $p_2=0.03$.

Este análisis mide la distorsión de entradas representativas del kernel. No se utiliza para seleccionar el mejor mapa ni para ajustar el clasificador.


In [ ]:

# 10. Ruido despolarizante sobre pares representativos
def make_noise_model(p1, p2):
    model = NoiseModel()
    model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ['sx', 'x'])
    model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx'])
    return model


def overlap_circuit(feature_map, x_i, x_j):
    params = list(feature_map.parameters)
    U_i = feature_map.assign_parameters(dict(zip(params, x_i)), inplace=False)
    U_j = feature_map.assign_parameters(dict(zip(params, x_j)), inplace=False)
    qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
    qc.compose(U_j, inplace=True)
    qc.compose(U_i.inverse(), inplace=True)
    qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))
    return qc


def noisy_overlap(feature_map, x_i, x_j, noise_model, shots, seed):
    backend = AerSimulator(noise_model=noise_model)
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        seed_transpiler=seed,
    )
    physical = pm.run(overlap_circuit(feature_map, x_i, x_j))
    result = backend.run(physical, shots=shots, seed_simulator=seed).result()
    counts = result.get_counts()
    return counts.get('0' * NUM_QUBITS, 0) / shots


def representative_pairs(labels, n_pairs, seed):
    rng = np.random.default_rng(seed)
    all_pairs = [(i, j) for i in range(len(labels)) for j in range(i + 1, len(labels))]
    intra = [(i, j) for i, j in all_pairs if labels[i] == labels[j]]
    inter = [(i, j) for i, j in all_pairs if labels[i] != labels[j]]
    n_intra = n_pairs // 2
    chosen_intra = [intra[k] for k in rng.choice(len(intra), size=n_intra, replace=False)]
    chosen_inter = [inter[k] for k in rng.choice(len(inter), size=n_pairs - n_intra, replace=False)]
    return chosen_intra + chosen_inter

noise_levels = {
    'mild': (0.001, 0.01),
    'moderate': (0.003, 0.03),
}

noise_rows = []
if RUN_NOISE_STUDY:
    for map_number, cfg in enumerate(MAP_CONFIGS):
        print('Noise:', cfg['name'])
        fmap = build_feature_map(cfg)
        pre = AnglePreprocessor(cfg['scaling']).fit(X_raw)
        X_angles = pre.transform(X_raw)
        K_exact = global_kernels[cfg['name']]
        pairs = representative_pairs(y, NOISE_PAIRS_PER_MAP, SEED + map_number)

        for level, (p1, p2) in noise_levels.items():
            model = make_noise_model(p1, p2)
            for pair_number, (i, j) in enumerate(pairs):
                t0 = time.perf_counter()
                estimate = noisy_overlap(
                    fmap, X_angles[i], X_angles[j], model,
                    NOISE_SHOTS, SEED + 1000 * map_number + pair_number,
                )
                elapsed = time.perf_counter() - t0
                noise_rows.append({
                    'map': cfg['name'],
                    'noise_level': level,
                    'i': i, 'j': j,
                    'pair_type': 'intra' if y[i] == y[j] else 'inter',
                    'exact_kernel': float(K_exact[i, j]),
                    'noisy_kernel': float(estimate),
                    'absolute_error': float(abs(estimate - K_exact[i, j])),
                    'simulation_seconds': elapsed,
                })

noise_results = pd.DataFrame(noise_rows)
if not noise_results.empty:
    noise_summary = (
        noise_results.groupby(['map', 'noise_level'])
        .agg(
            mean_absolute_error=('absolute_error', 'mean'),
            exact_mean=('exact_kernel', 'mean'),
            noisy_mean=('noisy_kernel', 'mean'),
            simulation_seconds=('simulation_seconds', 'sum'),
        )
        .reset_index()
    )
    display(noise_summary)


In [ ]:

# 11. Exportación y manifiesto reproducible
fold_results.to_csv(ARTIFACT_DIR / 'quantum_kernel_fold_metrics.csv', index=False)
summary.to_csv(ARTIFACT_DIR / 'quantum_kernel_cv_summary.csv', index=False)
geometry_results.to_csv(ARTIFACT_DIR / 'quantum_kernel_geometry.csv', index=False)
resource_results.to_csv(ARTIFACT_DIR / 'quantum_circuit_resources.csv', index=False)
if not shot_results.empty:
    shot_results.to_csv(ARTIFACT_DIR / 'quantum_shot_sensitivity.csv', index=False)
if not noise_results.empty:
    noise_results.to_csv(ARTIFACT_DIR / 'quantum_noise_sensitivity.csv', index=False)

# Matrices exactas de 64x64 para auditoría y visualización posterior.
np.savez_compressed(
    ARTIFACT_DIR / 'quantum_global_kernels.npz',
    **{name: matrix for name, matrix in global_kernels.items()}
)

best_row = summary.iloc[0].to_dict()
manifest = {
    'notebook': 'V05_Challenge2_Quantum_Kernel_Feature_Map_Study.ipynb',
    'seed': SEED,
    'num_samples': 64,
    'num_features': len(FEATURES),
    'num_qubits': NUM_QUBITS,
    'state_dimension': 2 ** NUM_QUBITS,
    'folds': 4,
    'outer_train_size': 48,
    'outer_validation_size': 16,
    'svc_kernel': 'precomputed',
    'kernel_definition': '|<phi(x_i)|phi(x_j)>|^2',
    'c_grid': list(C_GRID),
    'map_configs': MAP_CONFIGS,
    'best_map_by_mean_f1': best_row.get('map'),
    'best_mean_f1': best_row.get('f1_mean'),
    'quick_mode': QUICK_MODE,
    'shot_study': RUN_SHOT_STUDY,
    'noise_study': RUN_NOISE_STUDY,
    'data_file': str(raw_path),
    'fold_file': str(fold_path),
}
with open(ARTIFACT_DIR / 'quantum_kernel_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

checks = pd.DataFrame([
    {'check': '64 real samples', 'passed': len(raw64) == 64},
    {'check': 'balanced labels 32/32', 'passed': raw64[TARGET].value_counts().to_dict() == {0: 32, 1: 32}},
    {'check': 'four frozen folds', 'passed': set(validation_fold) == {0, 1, 2, 3}},
    {'check': 'nine features equal nine qubits', 'passed': len(FEATURES) == NUM_QUBITS},
    {'check': 'SVC uses precomputed kernel', 'passed': True},
    {'check': 'preprocessing fit inside each outer fold', 'passed': True},
])
checks.to_csv(ARTIFACT_DIR / 'quantum_kernel_checks.csv', index=False)
display(checks)

print('Mejor configuración por F1 medio:')
display(summary.head(1))
print('Archivos exportados en:', ARTIFACT_DIR)



## Interpretación final

Al ejecutar el notebook, la tabla `quantum_kernel_cv_summary.csv` debe usarse como resultado predictivo principal. Los mapas de calor, alineaciones y espectros ayudan a explicar **por qué** un mapa funciona o falla:

- Kernel casi identidad: estados casi ortogonales, baja generalización entre muestras.
- Kernel casi uniforme: todas las observaciones parecen similares, poca capacidad de separación.
- Buena estructura: mayor similitud intraclase que interclase, alineación positiva con las etiquetas y un espectro no degenerado.

El mejor mapa no se selecciona por una única matriz visual. Se selecciona por el F1 medio de los cuatro folds, acompañado por balanced accuracy, variabilidad, geometría y costo del circuito.

### Referencias de implementación

- Qiskit `ZZFeatureMap` y `PauliFeatureMap`.
- Definición de kernel de fidelidad: $K(x,y)=|\langle\phi(x)|\phi(y)\rangle|^2$.
- Integración con `SVC(kernel="precomputed")`.
- Qiskit Aer para ruido despolarizante y muestreo finito.
